In [1]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pickle
import traceback
import os
from pathlib import Path

from IPython.display import clear_output
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from aipad.pad_imputation import (
    convert_to_bool_coverage, load_wind_event, coverage_overlap, 
    load_random_file, pad_histogram
)
from aipad.spacecrafts import SoloConstants, WindConstants

solo = SoloConstants()
wind = WindConstants()

In [2]:
bin_width_deg = 1
time_avg_min = 1
avg_bin_dir = f"{time_avg_min}min_{bin_width_deg}deg"

load_path = Path("./data/intensities") / avg_bin_dir
cov_path = Path("./data/coverages") / avg_bin_dir
plot_path = Path(f"./plots/matrix_test/{time_avg_min}min_{bin_width_deg}deg")

In [3]:
from numpy.lib.stride_tricks import sliding_window_view
# n_train = 150
# n_test = 50

hist_list = []
reduced_hist_list = []
intensity_list = []

# TODO handle metadata
for file in load_path.iterdir():
    try:
        npz = np.load(file)
        hist = npz["full"]
        reduced_hist = npz["reduced"]
        intensity = npz["intensity_data"]
        hist_list.append(hist)
        reduced_hist_list.append(reduced_hist)
        intensity_list.append(intensity)
    except ValueError:
        print(traceback.format_exc())
        continue

X_train, X_test, y_train, y_test, I_train, I_test = train_test_split(hist_list, reduced_hist_list, intensity_list, random_state=123, shuffle=True)

# Concatenate contiguous rows to one feature vector (5 * 180 = 900 features)
trains = []
for train in X_train:
    train_stacked = sliding_window_view(train, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    trains.append(train_stacked)

X_train_stacked = np.vstack(trains)

In [4]:
X_train_stacked.shape

(127448, 900)

In [5]:
model_path = Path("./data/models")
model_path.mkdir(exist_ok=True)

neighbors = 5

model = KNNImputer(n_neighbors=neighbors, weights="distance", keep_empty_features=True)
model.fit(X_train_stacked)
dump_file = open(model_path / f'knnimputer_20250111', 'wb')
pickle.dump(model, dump_file)
dump_file.close()

Training data: every instance (angle vector / timestamp) is appended with the next 4 -> 900 features. This is then vstacked and fed into the model as array of shape (# of total timestamps, 900).
Test histograms are reshaped to contain 900 features. R2 score is calculated for each 5 x 180 block.

TODO: make reshaping stuff cleaner. Pipeline should also be more concise, I think (or wrapped somehow)

In [9]:
def form_test_matrices(model: KNNImputer, X_test: np.ndarray, y_test: np.ndarray) -> list:   # TODO think of a better name
    true = X_test
    test = y_test
    reduced = np.where(np.isfinite(test), true, np.nan)

    reduced_reshaped = reduced.reshape((144,900))
    pred_full_reshaped = model.transform(reduced_reshaped)
    pred_full = pred_full_reshaped.reshape((720,180))

    pred = np.where((np.isnan(reduced) 
                    & np.isfinite(true) 
                    & np.meshgrid(np.isfinite(reduced).any(axis=1), np.arange(0, reduced.shape[1]), indexing="ij")[0]), 
                    pred_full, np.nan)
    target = np.where(np.isfinite(pred), true, np.nan)

    return [true, reduced, pred_full, pred, target]

def calculate_scores(res: list) -> list:
    pred = res[3]
    target = res[4]
    pred_reshaped = pred.reshape((144,900))
    target_reshaped = target.reshape((144,900))
    scores = []
    mse_scores = []
    for i, _ in enumerate(pred_reshaped):
        if np.any(np.isfinite(target_reshaped[i])) and np.any(np.isfinite(pred_reshaped[i])):
            scores.append(r2_score(target_reshaped[i][np.isfinite(target_reshaped[i])], pred_reshaped[i][np.isfinite(pred_reshaped[i])]))
            mse_scores.append(-mean_squared_error(target_reshaped[i][np.isfinite(target_reshaped[i])], pred_reshaped[i][np.isfinite(pred_reshaped[i])]))
        else:
            scores.append(np.nan)
            mse_scores.append(np.nan)

    return [scores, mse_scores]

def plot_results(res: list, intensities: np.ndarray, save_plot=True, save_path=None) -> None:
    scores, mse_scores = calculate_scores(res)

    true, reduced, pred_full, pred, target = res

    fig, axs = plt.subplots(nrows=9, figsize=(16,36), sharex=True)

    X, Y = np.meshgrid(np.arange(0, true.shape[0]), np.arange(0, true.shape[1]), indexing="ij")

    norm = LogNorm(np.nanmin(true), np.nanmax(true))
    diff_norm = LogNorm(1e-3, 1e3)

    for i in range(8):
        axs[0].plot(intensities[:,i], label=wind.sectors[i])
    axs[0].set_title("Intensities")
    axs[0].set_yscale("log")
    axs[0].legend(loc="upper right")

    axs[1].pcolormesh(X, Y, true, norm=norm, cmap="inferno")
    axs[1].set_title("WIND PAD")

    axs[2].pcolormesh(X, Y, reduced, norm=norm, cmap="inferno")
    axs[2].set_title("WIND PAD w/ reduction")

    axs[3].pcolormesh(X, Y, pred_full, norm=norm, cmap="inferno")
    axs[3].set_title(f"k-NN imputed values, k = {neighbors}")

    axs[4].pcolormesh(X, Y, pred, norm=norm, cmap="inferno")
    axs[4].set_title("Predicted values")

    axs[5].pcolormesh(X, Y, target, norm=norm, cmap="inferno")
    axs[5].set_title("Target values")

    mesh = axs[6].pcolormesh(X, Y, target - pred, norm=diff_norm)
    axs[6].set_title("Difference")
    axins = inset_axes(axs[6], width="100%", height="100%", loc="center", 
                                bbox_to_anchor=(1.01,0,0.03,1), bbox_transform=axs[6].transAxes, borderpad=0.2)
    cbar = fig.colorbar(mesh, cax=axins, orientation="vertical")

    axs[7].plot(np.arange(0,720,5), scores)
    axs[7].set_title("R2 score per 5 x 180 block")

    axs[8].plot(np.arange(0,720,5), mse_scores)
    axs[8].set_title("MSE score per 5 x 180 block")

    if save_plot:
        plt.savefig(save_path)
        plt.close()

In [10]:
for j in range(0, len(y_test)):
    res = form_test_matrices(model, X_test[j], y_test[j])
    plot_results(res, I_test[j], save_path=plot_path / f"results_{j}.png")

KeyboardInterrupt: 